<a href="https://colab.research.google.com/github/UNOKOR/Day2Assignment/blob/master/Copy_of_Only_Muon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ResNet 모델 정의

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os
import time

import torchvision
import torchvision.transforms as transforms


# ============================================================
# 모델 정의
# ============================================================

# ResNet18을 위해 최대한 간단히 수정한 BasicBlock 클래스
class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()

        # 3x3 필터를 사용 (너비와 높이를 줄일 때는 stride 값 조절)
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)  # 배치 정규화(batch normalization)

        # 3x3 필터를 사용 (패딩을 1만큼 주기 때문에 너비와 높이가 동일)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)  # 배치 정규화(batch normalization)

        self.shortcut = nn.Sequential()  # identity인 경우
        # stride가 1이 아니거나, 채널 수(in_planes -> planes)가 바뀌는 경우 모두 1x1 conv shortcut 필요
        # (기존 코드는 stride만 체크해서 이번 설정에선 우연히 문제 없었지만, depth/채널 구성을 바꾸면 shape mismatch 위험 있었음)
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)  # (핵심) skip connection
        out = F.relu(out)
        return out


# ResNet 클래스 정의
class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=100):
        super(ResNet, self).__init__()
        self.in_planes = 64

        # 64개의 3x3 필터(filter)를 사용
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes  # 다음 레이어를 위해 채널 수 변경
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)  # 출력: [batch_size, 512, 4, 4]
        out = F.avg_pool2d(out, 4)  # 출력: [batch_size, 512, 1, 1]
        out = out.view(out.size(0), -1)  # 출력: [batch_size, 512]
        out = self.linear(out)
        return out


# ResNet18 함수 정의
def ResNet18():
    return ResNet(BasicBlock, [2, 2, 2, 2])

데이터 준비

In [ ]:
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD = (0.2675, 0.2565, 0.2761)
SPLIT_SEED = 42

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

# Google Drive (Colab 환경에서만 필요)
from google.colab import drive
drive.mount('/content/drive')

data_root = '/content/drive/MyDrive/cifar100_data'
os.makedirs(data_root, exist_ok=True)

optimizer_name = 'Muon'
checkpoint_dir = os.path.join('/content/drive/MyDrive/muon_cifar100_checkpoints', optimizer_name)
os.makedirs(checkpoint_dir, exist_ok=True)

latest_ckpt_path = os.path.join(checkpoint_dir, 'latest_resnet18_cifar100.pth')
best_ckpt_path = os.path.join(checkpoint_dir, 'best_resnet18_cifar100.pth')

full_train_dataset_aug = torchvision.datasets.CIFAR100(
    root=data_root, train=True, download=True, transform=transform_train
)
full_train_dataset_clean = torchvision.datasets.CIFAR100(
    root=data_root, train=True, download=True, transform=transform_test
)

train_dataset, _ = torch.utils.data.random_split(
    full_train_dataset_aug, [45000, 5000],
    generator=torch.Generator().manual_seed(SPLIT_SEED)
)
_, val_dataset = torch.utils.data.random_split(
    full_train_dataset_clean, [45000, 5000],
    generator=torch.Generator().manual_seed(SPLIT_SEED)
)

test_dataset = torchvision.datasets.CIFAR100(
    root=data_root, train=False, download=True, transform=transform_test
)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=100, shuffle=False, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=100, shuffle=False, num_workers=2)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


100%|██████████| 169M/169M [50:19<00:00, 56.0kB/s]


Optimizor_Muon

In [ ]:
def zeropower_via_newtonschulz5(G, steps=5, eps=1e-7):
    assert G.ndim == 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.bfloat16()
    X /= (X.norm() + eps)
    if G.size(0) > G.size(1):
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X
    if G.size(0) > G.size(1):
        X = X.T
    return X.to(G.dtype)


class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True, ns_steps=5):
        defaults = dict(lr=lr, momentum=momentum, nesterov=nesterov, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, momentum = group['lr'], group['momentum']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if g.ndim > 2:
                    g = g.view(g.size(0), -1)
                state = self.state[p]
                if 'momentum_buffer' not in state:
                    state['momentum_buffer'] = torch.zeros_like(g)
                buf = state['momentum_buffer']
                buf.mul_(momentum).add_(g)
                g = g.add(buf, alpha=momentum) if group['nesterov'] else buf
                u = zeropower_via_newtonschulz5(g, steps=group['ns_steps'])
                p.data.add_(u.view_as(p.data), alpha=-lr)
        return loss


class CombinedOptimizer:
    def __init__(self, optimizers):
        self.optimizers = optimizers

    @property
    def param_groups(self):
        return [pg for opt in self.optimizers for pg in opt.param_groups]

    def zero_grad(self, set_to_none=True):
        for opt in self.optimizers:
            opt.zero_grad(set_to_none=set_to_none)

    def step(self):
        for opt in self.optimizers:
            opt.step()

    def state_dict(self):
        return [opt.state_dict() for opt in self.optimizers]

    def load_state_dict(self, state_dicts):
        for opt, sd in zip(self.optimizers, state_dicts):
            opt.load_state_dict(sd)


def adjust_learning_rate(optimizer, epoch):
    decay = 1.0
    if epoch >= 30:
        decay /= 10
    if epoch >= 45:
        decay /= 10
    if epoch >= 60:
        decay /= 10
    for param_group in optimizer.param_groups:
        param_group['lr'] = param_group['initial_lr'] * decay

SAM - 일반화

In [ ]:
class SAM:
    def __init__(self, base_optimizer, rho=0.05, adaptive=False):
        assert rho >= 0
        self.base_optimizer = base_optimizer
        self.rho = rho
        self.adaptive = adaptive
        self.state = {}  # p -> {'old_p': tensor}

    @property
    def param_groups(self):
        return self.base_optimizer.param_groups

    def zero_grad(self, set_to_none=True):
        self.base_optimizer.zero_grad(set_to_none=set_to_none)

    @torch.no_grad()
    def _grad_norm(self):
        device = self.param_groups[0]['params'][0].device
        norms = [
            ((p.abs() if self.adaptive else 1.0) * p.grad).norm(p=2).to(device)
            for group in self.param_groups
            for p in group['params']
            if p.grad is not None
        ]
        return torch.norm(torch.stack(norms), p=2)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        # w -> w + e(w) : 손실이 최대로 커지는 방향으로 이동 (ascent step)
        grad_norm = self._grad_norm()
        scale = self.rho / (grad_norm + 1e-12)
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                self.state.setdefault(p, {})['old_p'] = p.data.clone()
                e_w = (torch.pow(p, 2) if self.adaptive else 1.0) * p.grad * scale
                p.add_(e_w)
        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        # w + e(w) -> w 로 복원한 뒤, perturbed 지점에서 구한 gradient로
        # base optimizer(Muon+SGD)가 실제 파라미터 업데이트 수행
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                p.data = self.state[p]['old_p']
        self.base_optimizer.step()
        if zero_grad:
            self.zero_grad()

    def state_dict(self):
        return self.base_optimizer.state_dict()

    def load_state_dict(self, state_dict):
        self.base_optimizer.load_state_dict(state_dict)


# BatchNorm running mean/var가 ascent step(1차 forward)에서 오염되는 것을 방지
# (공식 구현·후속 논문들에서 권장하는 관행)
def disable_running_stats(model):
    def _disable(module):
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.backup_momentum = module.momentum
            module.momentum = 0
    model.apply(_disable)

def enable_running_stats(model):
    def _enable(module):
        if isinstance(module, nn.modules.batchnorm._BatchNorm) and hasattr(module, 'backup_momentum'):
            module.momentum = module.backup_momentum
    model.apply(_enable)

학습 / 검증 / 테스트 루프

In [ ]:
def train(epoch, net, optimizer, criterion, train_loader, device):
    print('\n[ Train epoch: %d ]' % epoch)
    net.train()

    train_loss = 0
    correct = 0
    total = 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        current_correct = predicted.eq(targets).sum().item()
        correct += current_correct

        if batch_idx % 100 == 0:
            print('\nCurrent batch:', batch_idx)
            print('Current batch train accuracy:', current_correct / targets.size(0))
            print('Current batch train loss:', loss.item())

    train_acc = correct / total
    train_loss_avg = train_loss / total
    print('\nTotal average train accuracy:', train_acc)
    print('Total average train loss:', train_loss_avg)
    return train_acc

def train_sam(epoch, net, sam_optimizer, criterion, train_loader, device):
    print('\n[ Train epoch: %d ]' % epoch)
    net.train()
    train_loss = 0
    correct = 0
    total = 0
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        # 1st pass: 현재 w에서 gradient 계산 (정확도/로스 로깅도 이 pass 기준)
        enable_running_stats(net)
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        sam_optimizer.first_step(zero_grad=True)

        # 2nd pass: perturbed w+e(w)에서 다시 gradient 계산 → 실제 업데이트
        disable_running_stats(net)
        criterion(net(inputs), targets).backward()
        sam_optimizer.second_step(zero_grad=True)

        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        current_correct = predicted.eq(targets).sum().item()
        correct += current_correct
        if batch_idx % 100 == 0:
            print('\nCurrent batch:', batch_idx)
            print('Current batch train accuracy:', current_correct / targets.size(0))
            print('Current batch train loss:', loss.item())
    train_acc = correct / total
    train_loss_avg = train_loss / total
    print('\nTotal average train accuracy:', train_acc)
    print('Total average train loss:', train_loss_avg)
    return train_acc

def validate(epoch, net, criterion, val_loader, device):
    print('\n[ Validation epoch: %d ]' % epoch)
    net.eval()

    loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(val_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss += criterion(outputs, targets).item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    val_acc = correct / total
    val_loss = loss / total
    print('Validation accuracy:', val_acc)
    print('Validation loss:', val_loss)
    return val_acc


def test(epoch, net, criterion, test_loader, device):
    print('\n[ Test epoch: %d ]' % epoch)
    net.eval()

    loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(test_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss += criterion(outputs, targets).item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    print('\nTotal average test accuracy:', correct / total)
    print('Total average test loss:', loss / total)


모델 / 옵티마이저 초기화 (여기를 다시 실행하면 완전히 새로 시작)실행

In [ ]:
RESET_TRAINING = True  # 이어서 학습하려면 False로 바꾸세요
optimizer_name = 'Muon'
checkpoint_dir = os.path.join('/content/drive/MyDrive/muon_cifar100_checkpoints', optimizer_name)
os.makedirs(checkpoint_dir, exist_ok=True)
latest_ckpt_path = os.path.join(checkpoint_dir, 'latest_resnet18_cifar100.pth')
best_ckpt_path = os.path.join(checkpoint_dir, 'best_resnet18_cifar100.pth')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

net = ResNet18().to(device)
learning_rate = 0.1
criterion = nn.CrossEntropyLoss()

# 파라미터 분리: linear/BatchNorm/bias -> SGD, 나머지(conv weight, ndim>=2) -> Muon
muon_params, other_params = [], []
for name, p in net.named_parameters():
    if 'linear' in name:
        other_params.append(p)
    elif p.ndim >= 2:
        muon_params.append(p)
    else:
        other_params.append(p)

muon_optimizer = Muon(muon_params, lr=0.02, momentum=0.95)
other_optimizer = torch.optim.SGD(other_params, lr=learning_rate, momentum=0.9, weight_decay=5e-4)

# adjust_learning_rate가 참조하는 initial_lr 기록 (누락되면 KeyError 발생)
for pg in muon_optimizer.param_groups:
    pg['initial_lr'] = pg['lr']
for pg in other_optimizer.param_groups:
    pg['initial_lr'] = pg['lr']

optimizer = CombinedOptimizer([muon_optimizer, other_optimizer])
sam_optimizer = SAM(optimizer, rho=0.05, adaptive=False)

TOTAL_EPOCHS = 300
best_acc = 0.0
start_epoch = 0
best_epoch = 0
train_acc_history, val_acc_history = [], []

if not RESET_TRAINING and os.path.exists(latest_ckpt_path):
    print('기존 체크포인트를 발견했습니다. 이어서 학습합니다.')
    checkpoint = torch.load(latest_ckpt_path, map_location=device)
    net.load_state_dict(checkpoint['net'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    start_epoch = checkpoint['epoch'] + 1
    best_acc = checkpoint['best_acc']
    best_epoch = checkpoint.get('best_epoch', 0)
    train_acc_history = checkpoint.get('train_acc_history', [])
    val_acc_history = checkpoint.get('val_acc_history', [])
    print(f"epoch {start_epoch}부터 재개, 기존 best_acc = {best_acc:.4f}")
else:
    print('모델과 옵티마이저를 새로 초기화했습니다. 0번 에폭부터 시작합니다.')

Device: cuda
모델과 옵티마이저를 새로 초기화했습니다. 0번 에폭부터 시작합니다.


학습 루프

In [ ]:
start_time = time.time()

for epoch in range(start_epoch, TOTAL_EPOCHS):

    adjust_learning_rate(optimizer, epoch)
    train_acc = train(epoch, net, optimizer, criterion, train_loader, device)
    val_acc = validate(epoch, net, criterion, val_loader, device)

    train_acc_history.append(train_acc)
    val_acc_history.append(val_acc)

    # baseline(Muon-only) 체크포인트 저장: optimizer.state_dict() 사용 (SAM 버전은 optimizer_name='Muon_SAM'으로 별도 실행)
    latest_state = {
        'net': net.state_dict(),
        'optimizer': optimizer.state_dict(),
        'epoch': epoch,
        'val_acc': val_acc,
        'best_acc': best_acc,
        'best_epoch': best_epoch,
        'train_acc_history': train_acc_history,
        'val_acc_history': val_acc_history,
    }
    torch.save(latest_state, latest_ckpt_path)

    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch
        latest_state['best_acc'] = best_acc
        latest_state['best_epoch'] = best_epoch
        torch.save(latest_state, best_ckpt_path)
        print('Best model saved!')
        print('Best validation accuracy:', best_acc)

    print('\nTime elapsed:', time.time() - start_time)


[ Train epoch: 0 ]

Current batch: 0
Current batch train accuracy: 0.0078125
Current batch train loss: 4.7509660720825195

Current batch: 100
Current batch train accuracy: 0.109375
Current batch train loss: 3.7516353130340576

Current batch: 200
Current batch train accuracy: 0.234375
Current batch train loss: 3.133444309234619

Current batch: 300
Current batch train accuracy: 0.328125
Current batch train loss: 2.574673891067505

Total average train accuracy: 0.20422222222222222
Total average train loss: 3.2949963681962755

[ Validation epoch: 0 ]
Validation accuracy: 0.3404
Validation loss: 2.5105623149871827
Best model saved!
Best validation accuracy: 0.3404

Time elapsed: 64.45510077476501

[ Train epoch: 1 ]

Current batch: 0
Current batch train accuracy: 0.375
Current batch train loss: 2.236712694168091

Current batch: 100
Current batch train accuracy: 0.3671875
Current batch train loss: 2.53163480758667

Current batch: 200
Current batch train accuracy: 0.46875
Current batch train

결과 시각화 + best 모델 테스트

In [ ]:
import torch
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from google.colab import files
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_acc_history) + 1), train_acc_history, label="Train Accuracy")
plt.plot(range(1, len(val_acc_history) + 1), val_acc_history, label="Validation Accuracy")
plt.axvline(30, color="gray", linestyle="--", alpha=0.5)
plt.axvline(45, color="gray", linestyle="--", alpha=0.5)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Train vs Validation Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig("accuracy_curve.png")
plt.show()

checkpoint = torch.load(best_ckpt_path)
net.load_state_dict(checkpoint['net'])

print('Best epoch:', checkpoint['epoch'])
print('Best validation accuracy:', checkpoint['val_acc'])

test(checkpoint['epoch'], net, criterion, test_loader, device)



테스트

In [ ]:
# 1. 파일 업로드 (버튼이 뜨면 사진 파일을 선택하세요)
uploaded = files.upload()

# 업로드한 파일명 가져오기 (여러 장 올렸다면 첫 번째 파일 기준)
img_path = list(uploaded.keys())[0]

# 2. 이미지 열기 + 확인
raw_img = Image.open(img_path).convert('RGB')

plt.imshow(raw_img)
plt.axis('off')
plt.title('Uploaded image (original size)')
plt.show()

# 3. 학습 때와 동일한 전처리: 32x32로 리사이즈 + CIFAR-100 정규화
CIFAR100_MEAN = (0.5071, 0.4865, 0.4409)
CIFAR100_STD  = (0.2673, 0.2564, 0.2762)

transform_test = transforms.Compose([
    transforms.Resize((32, 32)),           # 업로드 사진은 보통 32x32가 아니므로 리사이즈 필수
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

resized_preview = raw_img.resize((32, 32))
plt.imshow(resized_preview)
plt.axis('off')
plt.title('Resized to 32x32 (model input)')
plt.show()

# 4. best 체크포인트 불러오기 (이미 로드되어 있다면 생략 가능)
checkpoint = torch.load(best_ckpt_path, map_location=device)
net.load_state_dict(checkpoint['net'])
net.eval()

# 5. 모델 입력 형태로 변환 후 예측
input_tensor = transform_test(raw_img).unsqueeze(0).to(device)  # (1, 3, 32, 32)

with torch.no_grad():
    output = net(input_tensor)
    probs = torch.softmax(output, dim=1)
    top5_prob, top5_idx = probs.topk(5, dim=1)


class_names = full_train_dataset_aug.classes

print('예측 결과 (Top-5):')
for rank in range(5):
    idx = top5_idx[0][rank].item()
    prob = top5_prob[0][rank].item()
    print(f'{rank+1}. {class_names[idx]} ({prob*100:.2f}%)')